# 05 — Préparation des features pour la modélisation
Objectif : identifier les features les plus informatives, détecter les redondances, et répondre aux décisions de conception ouvertes (D1–D3).

**Source :** PostgreSQL `hotel_features_full`.  
**Note :** mettre `FULL_TABLE = True` pour le calcul de l'information mutuelle sur la table complète (1,74 M lignes). Par défaut, un échantillon de 30% est utilisé.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from pathlib import Path
from sklearn.feature_selection import mutual_info_regression
import sys
sys.path.insert(0, "..")
from feature_engineering.config import POSTGRES_URI

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"savefig.dpi": 300, "figure.dpi": 120})

engine = create_engine(POSTGRES_URI)

FIGURES_DIR = Path("figures/05_feature_readiness")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FULL_TABLE = False  # Set True to score mutual information on full table

def save_fig(name: str) -> None:
    path = FIGURES_DIR / f"{name}.png"
    plt.savefig(path, bbox_inches="tight")
    print(f"Enregistré → {path}")

In [2]:
sample_pct = 100 if FULL_TABLE else 30

df = pd.read_sql(f"""
    SELECT
        price_per_night, nights, days_until_checkin, stars_int, adults,
        check_in_month, check_in_dow, check_in_quarter, check_in_week_of_year,
        is_weekend_checkin::int AS is_weekend_checkin,
        is_ramadan::int AS is_ramadan,
        is_tunisia_public_holiday::int AS is_tunisia_public_holiday,
        is_tunisia_school_holiday::int AS is_tunisia_school_holiday,
        is_school_holiday_france::int AS is_school_holiday_france,
        is_school_holiday_germany::int AS is_school_holiday_germany,
        is_school_holiday_uk::int AS is_school_holiday_uk,
        days_to_nearest_european_holiday,
        view_upgrade_count_offered,
        has_free_view_upgrade::int AS has_free_view_upgrade,
        is_supplement_variant::int AS is_supplement_variant,
        peer_tight_count, peer_tight_median,
        peer_medium_count, peer_medium_median, peer_medium_p25, peer_medium_p75,
        peer_medium_std,
        peer_loose_count, peer_loose_median,
        sur_demande::int AS sur_demande,
        sur_demande_rate_city_checkin, sur_demande_rate_city_stars_checkin,
        sur_demande_rate_city_stars_boarding_checkin,
        city_activity_count_checkin
    FROM hotel_features_full TABLESAMPLE BERNOULLI({sample_pct})
""", engine)

print(f"Chargé : {len(df):,} lignes, {df.shape[1]} colonnes")

MemoryError: Unable to allocate 1.07 GiB for an array with shape (24, 5961318) and data type int64

## 1. Corrélation avec le prix par nuit (Spearman)

In [ ]:
feature_cols = [c for c in df.columns if c != "price_per_night"]

corr_vals = {}
for col in feature_cols:
    series = pd.to_numeric(df[col], errors="coerce")
    paired = pd.concat([series, df["price_per_night"]], axis=1).dropna()
    if len(paired) < 100:
        continue
    r = paired.corr(method="spearman").iloc[0, 1]
    corr_vals[col] = r

corr_df = (pd.Series(corr_vals)
             .rename("spearman")
             .to_frame()
             .assign(abs_spearman=lambda d: d["spearman"].abs())
             .sort_values("abs_spearman", ascending=False)
             .reset_index()
             .rename(columns={"index": "feature"}))

top25 = corr_df.head(25)

fig, ax = plt.subplots(figsize=(10, 11))
colors = ["steelblue" if v >= 0 else "salmon" for v in top25["spearman"].iloc[::-1]]
ax.barh(top25["feature"].iloc[::-1], top25["abs_spearman"].iloc[::-1], color=colors)
ax.axvline(0.1, color="orange", linestyle="--", alpha=0.8, label="Seuil 0.10")
ax.set_xlabel("|Corrélation de Spearman| avec prix_par_nuit")
ax.set_title("Top 25 features — corrélation absolue avec le prix par nuit")
ax.legend()
plt.tight_layout()
save_fig("01_correlation_spearman")
plt.show()

corr_df.head(30)

## 2. Information mutuelle

In [ ]:
# Select columns with > 30% non-null rate for MI scoring
mi_cols = [c for c in feature_cols
           if pd.to_numeric(df[c], errors="coerce").notna().mean() > 0.3]

X = df[mi_cols].apply(pd.to_numeric, errors="coerce")
# Impute with column median (for MI scoring only — does not affect downstream)
X = X.apply(lambda s: s.fillna(s.median()))
y = df["price_per_night"].fillna(df["price_per_night"].median())

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_df = (pd.DataFrame({"feature": mi_cols, "mi_score": mi_scores})
           .sort_values("mi_score", ascending=False)
           .reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(mi_df["feature"].head(25).iloc[::-1],
        mi_df["mi_score"].head(25).iloc[::-1], color="teal")
ax.set_xlabel("Score d'information mutuelle")
ax.set_title("Top 25 features — information mutuelle avec prix_par_nuit")
plt.tight_layout()
save_fig("02_information_mutuelle")
plt.show()

mi_df.head(30)

## 3. Multicolinéarité — Facteur d'inflation de la variance (VIF)

In [ ]:
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print("statsmodels non installé — installez avec: pip install statsmodels")

if HAS_STATSMODELS:
    comp_cols = [c for c in [
        "peer_tight_median", "peer_medium_median", "peer_loose_median",
        "peer_medium_p25", "peer_medium_p75",
        "observed_delta_vs_peer_medium_median_pct", "observed_delta_vs_peer_tight_median_pct",
    ] if c in df.columns]

    vif_data = df[comp_cols].apply(pd.to_numeric, errors="coerce").dropna()
    vif_results = []
    for i, col in enumerate(comp_cols):
        try:
            v = variance_inflation_factor(vif_data.values, i)
        except Exception:
            v = float("nan")
        vif_results.append({"feature": col, "VIF": v})

    vif_df = pd.DataFrame(vif_results).sort_values("VIF", ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["crimson" if v > 10 else "steelblue" for v in vif_df["VIF"]]
    ax.bar(vif_df["feature"], vif_df["VIF"], color=colors)
    ax.axhline(10, color="orange", linestyle="--", linewidth=1.5, label="Seuil VIF=10")
    ax.set_ylabel("VIF")
    ax.set_title("Facteur d'inflation de la variance — bloc features compétitives")
    ax.set_xticklabels(vif_df["feature"], rotation=35, ha="right")
    ax.legend()
    plt.tight_layout()
    save_fig("03_vif_features_competitives")
    plt.show()
    display(vif_df)

## 4. Tableau récapitulatif par groupe de features

In [ ]:
# Build lookup from computed results
spearman_lookup = corr_df.set_index("feature")["spearman"].to_dict()
mi_lookup = mi_df.set_index("feature")["mi_score"].to_dict()

groups = [
    {
        "Groupe": "Taxonomie produit",
        "Features clés": "boarding_canonical, stars_int, nights, adults",
        "Couverture": "100%",
        "Features repr.": ["stars_int", "nights"],
    },
    {
        "Groupe": "Fenêtre de réservation",
        "Features clés": "days_until_checkin",
        "Couverture": "100%",
        "Features repr.": ["days_until_checkin"],
    },
    {
        "Groupe": "Compétitif (medium)",
        "Couverture": "95.6%",
    },
    {
        "Groupe": "Calendrier",
        "Features clés": "check_in_month, is_ramadan, is_school_holiday_france/de/uk",
        "Couverture": "100%",
        "Features repr.": ["check_in_month", "is_ramadan", "is_school_holiday_france"],
    },
    {
        "Groupe": "Proxy demande",
        "Features clés": "sur_demande_rate_city_checkin",
        "Couverture": "99.96%",
        "Features repr.": ["sur_demande_rate_city_checkin"],
    },
    {
        "Groupe": "Détail chambre",
        "Features clés": "room_view, room_tier, room_occupancy (via room_name)",
        "Couverture": "32–50%",
        "Features repr.": [],
    },
]

rows = []
for g in groups:
    sp_vals = [spearman_lookup.get(f, np.nan) for f in g["Features repr."]]
    mi_vals = [mi_lookup.get(f, np.nan) for f in g["Features repr."]]
    rows.append({
        "Groupe": g["Groupe"],
        "Features clés": g["Features clés"],
        "Couverture": g["Couverture"],
        "|Spearman| moy.": f"{np.nanmean(sp_vals):.3f}" if sp_vals else "—",
        "MI moy.": f"{np.nanmean(mi_vals):.3f}" if mi_vals else "—",
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

## 5. Décisions de modélisation ouvertes (D1–D3)

### D1 — Type de modèle
Le profil des features favorise **CatBoost** :
- Variables catégorielles à haute cardinalité (`boarding_canonical`, `city_name`, `room_base`) — CatBoost les gère nativement sans one-hot encoding.
- Features compétitives corrélées entre elles (VIF élevé attendu) — les arbres symétriques de CatBoost sont plus robustes à la multicolinéarité.
- `days_until_checkin` est non linéaire — les arbres de décision le capturent naturellement.

**LightGBM** reste obligatoire comme baseline de comparaison (cf. principe « baselines before ML »).

### D2 — Sorties probabilistes
Le rapport P90/P10 mesuré dans le NB03 justifie des sorties probabilistes (P10/P50/P90).  
CatBoost et LightGBM supportent la perte quantile nativement.  
→ **Décision : sorties P10/P50/P90** (un modèle par quantile ou utilisation de `MultiQuantileLoss` de CatBoost).

### D3 — Détection d'anomalies
Approche recommandée : **intervalles quantiles** du modèle P10/P90 comme seuils d'anomalie.  
Un prix observé en dehors de [P10, P90] prédit est flagué.  
Complément optionnel : Isolation Forest sur les résidus normalisés pour capturer les outliers structurels (hôtels isolés sans pair fiable).

## Conclusions pour la modélisation

### Features retenues
| Feature | Justification |
|---|---|
| `boarding_canonical` | Corrélation et MI élevés ; variable catégorielle principale |
| `stars_int` | Signal fort, couverture 100% |
| `nights` | Décote multi-nuits démontrée (NB03) |
| `days_until_checkin` | Effet non linéaire fort |
| `check_in_month` | Saisonnalité forte |
| `city_name` | Hétérogénéité géographique |
| `peer_medium_median` | Meilleur proxy du niveau de marché |
| `observed_delta_vs_peer_medium_median_pct` | Position relative dans le marché |
| `is_ramadan` | Effet calendaire significatif (NB04) |
| `sur_demande_rate_city_checkin` | Proxy de tension (sous réserve de l'audit NB04) |
| `adults` | Dimension produit |

### Features à transformer
| Feature | Transformation |
|---|---|
| `days_until_checkin` | Conserver brut + buckets (non-linéarité) |
| `peer_medium_median` | Log si distribution asymétrique |
| `city_activity_count_checkin` | Log (count variable) |

### Features exclues ou à usage limité
| Feature | Raison |
|---|---|
| `room_view`, `room_tier`, `room_occupancy` | Couverture < 50% ; utiles uniquement pour le groupe pair *tight* |
| `peer_tight_median` | Redondant avec `peer_medium_median` ; VIF élevé probable |
| `is_weekend_checkin` | Effet quasi nul (NB01) |
| `city_id` | Identifiant source-local, non cross-source |
| `scrape_run_id`, `scraped_at` | Identifiants scraping — ne doivent pas entrer dans le modèle |